# Figure 2b rerun — β-correlation clustering

**Panel 2b caption (from manuscript):** Perturbation-effect matrix from the ElasticNet model. Rows represent high-variance RNA features; columns represent retained gene-target perturbations; color indicates the β coefficient. Target-target Pearson correlations of β columns and RNA feature-feature Pearson correlations of β rows are shown below, with black boxes indicating K-means cluster boundaries.

Same β matrix as `20260417_Figure2_Nature.ipynb`. K-means is run on **β correlation** (matching `figure2_moi1/` and the co-scientist prompt's Input Data 1/4) — not signed-significance correlation.

**Outputs (all in `figures/nature_figures/Fig2/`):**
- `Fig2B_beta_matrix.{png,pdf}`
- `Fig2B_target_correlation.{png,pdf}` — target-target Pearson, 7 modules
- `Fig2B_feature_correlation.{png,pdf}` — feature-feature Pearson, 9 programs
- `Fig2B_cells_per_target_high.{png,pdf}` / `Fig2B_cells_per_target_low.{png,pdf}` — High/Low enrichment side strips
- `Fig2B_target_modules.csv`, `Fig2B_gene_programs.csv` — assignments (feed panel 2c lists & panel 2e Sankey)

Final cells verify modules match `results/figure2_moi1/target_clusters_moi1.csv` byte-for-byte and confirm IRF1 → HLA-II antigen-presentation regulators.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.cluster import KMeans

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['pdf.use14corefonts'] = True
warnings.filterwarnings('ignore')

BASE       = '/home/wangh256/hanchen/Pert_PG/perturb-me/PerturbME_transfer/PerturbCITE_ICR/202008_full_exp'
LM_DIR     = os.path.join(BASE, 'CROP/linear_model/17_cells_per_target/all_features')
MAGECK_DIR = os.path.join(BASE, 'guide_seq/mageck_out')
OUT_DIR    = '/gnet/is1/p01/shares/regevlab/hanchen/Pert_PG/perturb-me/figures/nature_figures/Fig2'
REF_DIR    = '/gnet/is1/p01/shares/regevlab/hanchen/Pert_PG/perturb-me/results/figure2_moi1'
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(name):
    plt.savefig(os.path.join(OUT_DIR, f'{name}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUT_DIR, f'{name}.png'), bbox_inches='tight', dpi=300)

In [ ]:
cov_names     = np.load(os.path.join(LM_DIR, 'cov_names.npy'))
feature_names = np.load(os.path.join(LM_DIR, 'feature_names.npy'))
B_mat         = np.load(os.path.join(LM_DIR, 'EN_B_EM.npy'))
print(f'Beta matrix: {B_mat.shape[0]} features x {B_mat.shape[1]} covariates')

In [ ]:
# Sparse filter — identical params to figure2_moi1 nb and Fig2 Nature nb
def remove_sparse(mat, rows, cols, approx_zero=0.05, row_thresh=0.25, col_thresh=0.8):
    row_sp = np.mean(np.abs(mat) <= approx_zero, axis=1)
    col_sp = np.mean(np.abs(mat) <= approx_zero, axis=0)
    rmask = row_sp < row_thresh
    cmask = col_sp < col_thresh
    return mat[rmask][:, cmask], rows[rmask], cols[cmask]

filt_B, filt_features, filt_covs = remove_sparse(B_mat, feature_names, cov_names)
print(f'After filter: {filt_B.shape[0]} features x {filt_B.shape[1]} covariates')

In [ ]:
# β correlation + K-means clustering
def corr_mat(M):
    C = pd.DataFrame(M).corr().to_numpy()
    C[np.isnan(C)] = 0
    return C

def cluster_by_kmeans(M, k, random_state=3):
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(M)
    order = np.concatenate([np.where(km.labels_ == c)[0] for c in range(k)])
    return order, km.labels_[order], km.labels_

K_TARGETS, K_PROGRAMS = 7, 9

B_corr_targets  = corr_mat(filt_B)
B_corr_features = corr_mat(filt_B.T)

tgt_order,  tgt_labels_sorted,  tgt_labels  = cluster_by_kmeans(B_corr_targets,  K_TARGETS)
feat_order, feat_labels_sorted, feat_labels = cluster_by_kmeans(B_corr_features, K_PROGRAMS)

print(f'Target modules ({K_TARGETS}) sizes:', [int((tgt_labels==c).sum()) for c in range(K_TARGETS)])
print(f'Gene programs  ({K_PROGRAMS}) sizes:', [int((feat_labels==c).sum()) for c in range(K_PROGRAMS)])

In [ ]:
# Save assignments
pd.DataFrame({'target': filt_covs, 'target_module': tgt_labels}) \
    .sort_values(['target_module','target']) \
    .to_csv(os.path.join(OUT_DIR, 'Fig2B_target_modules.csv'), index=False)
pd.DataFrame({'gene': filt_features, 'gene_program': feat_labels}) \
    .sort_values(['gene_program','gene']) \
    .to_csv(os.path.join(OUT_DIR, 'Fig2B_gene_programs.csv'), index=False)
print('Wrote Fig2B_target_modules.csv and Fig2B_gene_programs.csv')

In [ ]:
# β matrix re-ordered by modules x programs
plot_B = filt_B[feat_order, :][:, tgt_order]
fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_B, aspect=(plot_B.shape[1]/plot_B.shape[0]),
               cmap='bwr', clim=[-1, 1], interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both'); cbar.set_label('Beta coefficient'); cbar.minorticks_on()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel('Covariates (perturbation targets)')
ax.set_ylabel('Features (genes)')
ax.set_ylim([plot_B.shape[0]-0.5, -0.5])
save_fig('Fig2B_beta_matrix')
plt.show()

In [ ]:
# Target-target correlation + module boundary boxes
plot_corr_t = B_corr_targets[tgt_order, :][:, tgt_order]
fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr_t, aspect=(plot_corr_t.shape[1]/plot_corr_t.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1], interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both'); cbar.set_label('Pearson correlation'); cbar.minorticks_on()
ax.set_xlabel('Covariate'); ax.set_ylabel('Covariate')
ax.set_ylim([plot_corr_t.shape[0]-0.5, -0.5])

# Module boundaries + cluster ID labels along bottom
bounds = np.concatenate([[0], np.where(np.diff(tgt_labels_sorted) != 0)[0] + 1, [len(tgt_labels_sorted)]])
for s, e in zip(bounds[:-1], bounds[1:]):
    ax.add_patch(plt.Rectangle((s-0.5, s-0.5), e-s, e-s, fill=False, ec='k', lw=0.7))
midpoints = (bounds[:-1] + bounds[1:]) / 2
ax.set_xticks(midpoints); ax.set_xticklabels([f'#{i}' for i in range(K_TARGETS)], fontsize=9)
ax.set_yticks([])
save_fig('Fig2B_target_correlation')
plt.show()

In [ ]:
# Feature-feature correlation + program boundary boxes
plot_corr_f = B_corr_features[feat_order, :][:, feat_order]
fig, ax = plt.subplots(figsize=(6.4, 4.8))
im = ax.imshow(plot_corr_f, aspect=(plot_corr_f.shape[1]/plot_corr_f.shape[0]),
               cmap='PRGn', clim=[-0.1, 0.1], interpolation='nearest', rasterized=True)
cbar = fig.colorbar(im, ax=ax, extend='both'); cbar.set_label('Pearson correlation'); cbar.minorticks_on()
ax.set_xlabel('Feature'); ax.set_ylabel('Feature')
ax.set_ylim([plot_corr_f.shape[0]-0.5, -0.5])

bounds = np.concatenate([[0], np.where(np.diff(feat_labels_sorted) != 0)[0] + 1, [len(feat_labels_sorted)]])
for s, e in zip(bounds[:-1], bounds[1:]):
    ax.add_patch(plt.Rectangle((s-0.5, s-0.5), e-s, e-s, fill=False, ec='k', lw=0.7))
midpoints = (bounds[:-1] + bounds[1:]) / 2
ax.set_xticks(midpoints); ax.set_xticklabels([f'#{i}' for i in range(K_PROGRAMS)], fontsize=9)
ax.set_yticks([])
save_fig('Fig2B_feature_correlation')
plt.show()

In [ ]:
# High/Low enrichment side strips (cells per target, ordered by tgt_order)
all_genes_mg = np.load(os.path.join(MAGECK_DIR, 'all_genes.npy'), allow_pickle=True).astype(str)
cells_high   = np.load(os.path.join(MAGECK_DIR, 'cells_per_gene_high.npy'))
cells_low    = np.load(os.path.join(MAGECK_DIR, 'cells_per_gene_low.npy'))
idx = {g: i for i, g in enumerate(all_genes_mg)}

ordered_covs = filt_covs[tgt_order]
high_arr = np.array([cells_high[idx[t]] if t in idx else 0 for t in ordered_covs])
low_arr  = np.array([cells_low[idx[t]]  if t in idx else 0 for t in ordered_covs])

for arr, name, cmap_name in [(high_arr, 'Fig2B_cells_per_target_high', 'Reds'),
                              (low_arr,  'Fig2B_cells_per_target_low',  'Blues')]:
    fig, ax = plt.subplots(figsize=(0.6, 4.8))
    ax.imshow(arr.reshape(-1, 1), aspect=(1/80), cmap=cmap_name, clim=[0, 30], interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    save_fig(name)
    plt.show()

## Verification — modules match `figure2_moi1/` and IRF1 lands in HLA-II antigen-presentation regulators

In [ ]:
tm = pd.read_csv(os.path.join(OUT_DIR, 'Fig2B_target_modules.csv'))
ref = pd.read_csv(os.path.join(REF_DIR, 'target_clusters_moi1.csv')).rename(columns={'cluster':'cluster_ref'})
new = tm.rename(columns={'target_module':'cluster_new'})
merged = ref.merge(new, on='target')
ct = pd.crosstab(merged['cluster_ref'], merged['cluster_new'])
print('Cross-tab (figure2_moi1 ref vs Fig2B new):')
print(ct)
is_permutation = ((ct.values > 0).sum(axis=0).max() == 1) and ((ct.values > 0).sum(axis=1).max() == 1)
print(f'\nPartitions identical (up to label permutation): {is_permutation}')
assert len(merged) == len(ref) == len(new), f'size mismatch: ref={len(ref)} new={len(new)} merged={len(merged)}'

In [ ]:
irf1_mod = int(tm.loc[tm['target'] == 'IRF1', 'target_module'].iloc[0])
members  = sorted(tm.loc[tm['target_module'] == irf1_mod, 'target'].tolist())
key_hla2 = ['CIITA', 'NLRC5', 'RFX5', 'RFXANK', 'RFXAP', 'SPPL3']
key_jak  = ['JAK1', 'JAK2', 'STAT1', 'IFNGR1', 'IFNGR2']
print(f'IRF1 -> target_module {irf1_mod} ({len(members)} members)')
print(f'HLA-II regulator co-members present: {[g for g in key_hla2 if g in members]}')
print(f'JAK/STAT/IFN co-members present (should be []): {[g for g in key_jak  if g in members]}')
print(f'\nFirst 20 members: {members[:20]}')